In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build compact meta dataframe
def build_meta_df(df, name_col, top_n_names=50):
    rows = len(df)
    cmdb_unique = df['cmdb_id'].dropna().unique()
    cmdb_count = len(cmdb_unique)
    # Represent cmdb ids as comma-joined string limited to 50 entries
    cmdb_ids_str = ", ".join(map(str, cmdb_unique[:top_n_names]))
    ts_min = df['timestamp'].min() if 'timestamp' in df.columns else pd.NaT
    ts_max = df['timestamp'].max() if 'timestamp' in df.columns else pd.NaT
    meta = pd.DataFrame([{
        'file_rows': rows,
        'unique_cmdb_count': cmdb_count,
        'unique_cmdb_ids_sample': cmdb_ids_str,
        'global_timestamp_min_utc': ts_min,
        'global_timestamp_max_utc': ts_max
    }])
    # top name counts
    if name_col in df.columns:
        name_counts = df[name_col].value_counts().head(top_n_names).rename_axis(name_col).reset_index(name='count')
    else:
        name_counts = pd.DataFrame(columns=[name_col, 'count'])
    return meta, name_counts

# Metric summary
metric_meta, metric_kpi_counts = build_meta_df(metric, 'kpi_name', top_n_names=50)

# Trace summary
trace_meta, trace_trace_counts = build_meta_df(trace, 'trace_name', top_n_names=50)

# Log summary
log_meta, log_log_counts = build_meta_df(log, 'log_name', top_n_names=50)

# Error logs summary: different requirements (top 5 messages)
error_rows = len(error_logs)
error_cmdb_unique = error_logs['cmdb_id'].dropna().unique()
error_cmdb_count = len(error_cmdb_unique)
error_cmdb_ids_str = ", ".join(map(str, error_cmdb_unique[:50]))
error_ts_min = error_logs['timestamp'].min() if 'timestamp' in error_logs.columns else pd.NaT
error_ts_max = error_logs['timestamp'].max() if 'timestamp' in error_logs.columns else pd.NaT
# Top 5 messages with counts (handle missing 'message' column)
if 'message' in error_logs.columns:
    error_top_messages = error_logs['message'].value_counts().head(5).rename_axis('message').reset_index(name='count')
else:
    error_top_messages = pd.DataFrame(columns=['message', 'count'])

error_meta = pd.DataFrame([{
    'file_rows': error_rows,
    'unique_cmdb_count': error_cmdb_count,
    'unique_cmdb_ids_sample': error_cmdb_ids_str,
    'global_timestamp_min_utc': error_ts_min,
    'global_timestamp_max_utc': error_ts_max
}])

# Keep outputs compact: show meta summaries and top name/count tables (limited)
metric_meta, metric_kpi_counts, trace_meta, trace_trace_counts, log_meta, log_log_counts, error_meta, error_top_messages

```
Out[1]:
```
```python
# Build a plain-English summary string from previously loaded summaries
metric_rows = int(metric_meta.at[0, 'file_rows'])
metric_cmdb_sample = metric_meta.at[0, 'unique_cmdb_ids_sample']
metric_cmdb_count = int(metric_meta.at[0, 'unique_cmdb_count'])
metric_ts_min = metric_meta.at[0, 'global_timestamp_min_utc']
metric_ts_max = metric_meta.at[0, 'global_timestamp_max_utc']
metric_top_kpis = ", ".join(f"{r[kpi]}({r['count']})" for r in metric_kpi_counts.head(8).to_dict('records') if metric_kpi_counts.shape[0]>0 for kpi in ['kpi_name'])

trace_rows = int(trace_meta.at[0, 'file_rows'])
trace_cmdb_sample = trace_meta.at[0, 'unique_cmdb_ids_sample']
trace_cmdb_count = int(trace_meta.at[0, 'unique_cmdb_count'])
trace_ts_min = trace_meta.at[0, 'global_timestamp_min_utc']
trace_ts_max = trace_meta.at[0, 'global_timestamp_max_utc']
trace_top = ", ".join(f"{r['trace_name']}({r['count']})" for r in trace_trace_counts.head(6).to_dict('records'))

log_rows = int(log_meta.at[0, 'file_rows'])
log_cmdb_sample = log_meta.at[0, 'unique_cmdb_ids_sample']
log_cmdb_count = int(log_meta.at[0, 'unique_cmdb_count'])
log_ts_min = log_meta.at[0, 'global_timestamp_min_utc']
log_ts_max = log_meta.at[0, 'global_timestamp_max_utc']
log_top = ", ".join(f"{r['log_name']}({r['count']})" for r in log_log_counts.head(6).to_dict('records'))

error_rows = int(error_meta.at[0, 'file_rows'])
error_cmdb_count = int(error_meta.at[0, 'unique_cmdb_count'])
error_ts_min = error_meta.at[0, 'global_timestamp_min_utc']
error_ts_max = error_meta.at[0, 'global_timestamp_max_utc']
if not error_top_messages.empty:
    error_top = ", ".join(f"\"{r['message']}\"({r['count']})" for r in error_top_messages.to_dict('records'))
else:
    error_top = "No messages (file empty)"

# Format timestamps safely
def fmt(ts):
    return ts.isoformat() if ts is not pd.NaT and pd.notnull(ts) else "N/A"

summary = (
    "Summary of telemetry files (UTC):\n\n"
    f"metric.csv:\n"
    f"- rows: {metric_rows}\n"
    f"- unique cmdb count: {metric_cmdb_count}\n"
    f"- cmdb sample: {metric_cmdb_sample}\n"
    f"- top KPIs (name(count)) [top entries]: {metric_top_kpis}\n"
    f"- global timestamp range: {fmt(metric_ts_min)}  to  {fmt(metric_ts_max)}\n\n"
    f"trace.csv:\n"
    f"- rows: {trace_rows}\n"
    f"- unique cmdb count: {trace_cmdb_count}\n"
    f"- cmdb sample: {trace_cmdb_sample}\n"
    f"- top trace names (name(count)) [top entries]: {trace_top}\n"
    f"- global timestamp range: {fmt(trace_ts_min)}  to  {fmt(trace_ts_max)}\n\n"
    f"log.csv:\n"
    f"- rows: {log_rows}\n"
    f"- unique cmdb count: {log_cmdb_count}\n"
    f"- cmdb sample: {log_cmdb_sample}\n"
    f"- top log names (name(count)) [top entries]: {log_top}\n"
    f"- global timestamp range: {fmt(log_ts_min)}  to  {fmt(log_ts_max)}\n\n"
    f"error_logs.csv:\n"
    f"- rows: {error_rows}\n"
    f"- unique cmdb count: {error_cmdb_count}\n"
    f"- global timestamp range: {fmt(error_ts_min)}  to  {fmt(error_ts_max)}\n"
    f"- top messages (message(count)) [top 5]: {error_top}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_rows  unique_cmdb_count                             unique_cmdb_ids_sample  global_timestamp_min_utc  global_timestamp_max_utc
0       1850                 12  adservice, cartservice, checkoutservice, curre... 2024-01-17 20:58:00+00:00 2024-01-17 21:22:00+00:00,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    175
7       error     75,    file_rows  unique_cmdb_count                             unique_cmdb_ids_sample  global_timestamp_min_utc  global_timestamp_max_utc
0       3800                  8  checkoutservice, currencyservice, emailservice... 2024-01-17 20:58:00+00:00 2024-01-17 21:22:00+00:00,                                         trace_name  count
0                    trace.from_root.duration_mean    175
1                     trace.from_root.duration_p95    175
2                       trace.from_root.error_rate    175
3                        trace.from_root.row_count    175
4             trace.from_checkoutservice.row_count    125
..                                             ...    ...
39          trace.to_frontendservice.duration_mean     50
40  trace.from_recommendationservice.duration_mean     50
41   trace.from_recommendationservice.duration_p95     50
42     trace.from_recommendationservice.error_rate     50
43      trace.from_recommendationservice.row_count     50

[44 rows x 2 columns],    file_rows  unique_cmdb_count                             unique_cmdb_ids_sample  global_timestamp_min_utc  global_timestamp_max_utc
0        458                 10  adservice, cartservice, checkoutservice, curre... 2024-01-17 20:58:00+00:00 2024-01-17 21:22:00+00:00,           log_name  count
0  log.error_count    229
1    log.row_count    229,    file_rows  unique_cmdb_count unique_cmdb_ids_sample global_timestamp_min_utc global_timestamp_max_utc
0          0                  0                                             NaT                      NaT, Empty DataFrame
Columns: [message, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Load metric.csv and parse timestamps as UTC datetimes
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# Restrict to candidate services
metric_candidates = metric[metric['cmdb_id'].isin(candidates)].copy()

# Compute global statistics per (cmdb_id, kpi_name) BEFORE any time filtering
stats = metric_candidates.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    p50=lambda x: x.quantile(0.50),
    p95=lambda x: x.quantile(0.95),
    p05=lambda x: x.quantile(0.05),
    max='max'
).reset_index()

# Define incident window (UTC)
start = pd.to_datetime("2024-01-17 20:55:32", utc=True)
end = pd.to_datetime("2024-01-17 21:25:32", utc=True)

# Filter metric rows to the incident window
metric_window = metric_candidates[(metric_candidates['timestamp'] >= start) & (metric_candidates['timestamp'] <= end)].copy()

# Merge window rows with global stats to evaluate anomalies
merged = metric_window.merge(stats[['cmdb_id','kpi_name','p95','p05']], on=['cmdb_id','kpi_name'], how='left')

# Mark high/low anomalies using global thresholds
merged['high'] = merged['value'] >= merged['p95']
merged['low'] = merged['value'] <= merged['p05']

# Aggregate incident-window summaries per pair
incident_agg = merged.groupby(['cmdb_id','kpi_name']).agg(
    incident_points=('value','size'),
    high_anomaly_count=('high','sum'),
    low_anomaly_count=('low','sum'),
).reset_index()

# Earliest timestamps for high/low anomalies in the window
earliest_high = merged[merged['high']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index(name='earliest_high_anom_utc')
earliest_low = merged[merged['low']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index(name='earliest_low_anom_utc')

# Combine stats + incident aggregates + earliest times
result = stats.merge(incident_agg, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(earliest_high, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(earliest_low, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for incident-related numeric columns with zeros (no data in window)
result['incident_points'] = result['incident_points'].fillna(0).astype(int)
result['high_anomaly_count'] = result['high_anomaly_count'].fillna(0).astype(int)
result['low_anomaly_count'] = result['low_anomaly_count'].fillna(0).astype(int)

# Keep only requested columns and rename 'count' -> 'count' (global series count)
final_cols = [
    'cmdb_id', 'kpi_name', 'count', 'p50', 'p95', 'p05', 'max',
    'incident_points', 'high_anomaly_count', 'low_anomaly_count',
    'earliest_high_anom_utc', 'earliest_low_anom_utc'
]
final_df = result[final_cols].copy()

# Sort to surface most suspicious series first: high anomalies desc, then incident points desc
final_df_sorted = final_df.sort_values(by=['high_anomaly_count','incident_points','low_anomaly_count','max'], ascending=[False, False, False, False])

# Limit to top 50 rows for compactness
final_df_top50 = final_df_sorted.head(50).reset_index(drop=True)

# Aggregate summary: how many distinct (cmdb_id,kpi_name) pairs have high_anomaly_count>0 and low_anomaly_count>0
pairs_with_high = int((final_df['high_anomaly_count'] > 0).sum())
pairs_with_low = int((final_df['low_anomaly_count'] > 0).sum())
agg_summary = pd.DataFrame([{
    'distinct_pairs_total': len(final_df),
    'pairs_with_high_anomaly_count_gt0': pairs_with_high,
    'pairs_with_low_anomaly_count_gt0': pairs_with_low
}])

# Display compact results: top table and aggregate summary
final_df_top50, agg_summary

```
Out[2]:
```
```python
# Build a concise plain-English summary from previous results in memory:
# Uses final_df_top50 (top results table) and agg_summary (aggregate counts)

top_df = final_df_top50.copy()
agg = agg_summary.copy().iloc[0]

# Key aggregate numbers
total_pairs = int(agg['distinct_pairs_total'])
pairs_with_high = int(agg['pairs_with_high_anomaly_count_gt0'])
pairs_with_low = int(agg['pairs_with_low_anomaly_count_gt0'])

# Pick top 5 most suspicious series by high_anomaly_count, then incident_points
top5 = top_df.sort_values(by=['high_anomaly_count','incident_points'], ascending=[False, False]).head(5)

# Format top5 entries concisely
top5_lines = []
for _, r in top5.iterrows():
    ch = r['cmdb_id']
    kpi = r['kpi_name']
    ip = int(r['incident_points'])
    hi = int(r['high_anomaly_count'])
    lo = int(r['low_anomaly_count'])
    eh = r['earliest_high_anom_utc']
    el = r['earliest_low_anom_utc']
    eh_s = eh.isoformat() if pd.notnull(eh) else "N/A"
    el_s = el.isoformat() if pd.notnull(el) else "N/A"
    top5_lines.append(f"- {ch} / {kpi}: incident_points={ip}, high={hi}, low={lo}, first_high={eh_s}, first_low={el_s}")

# Build final summary string
summary = (
    f"Incident-window anomaly summary (UTC):\n\n"
    f"- Examined distinct (cmdb_id, kpi_name) pairs: {total_pairs}\n"
    f"- Pairs with at least one high anomaly (value >= P95): {pairs_with_high}\n"
    f"- Pairs with at least one low anomaly (value <= P05): {pairs_with_low}\n\n"
    f"Observations:\n"
    f"1) All {total_pairs} examined pairs show both high and low anomaly counts > 0 within the incident window.\n"
    f"2) Several KPI series (notably the 'socket' KPI across multiple services) have every window point flagged as both high and low. "
    f"This indicates constant-valued series where global P95 == P05 == observed value, causing both conditions to be true.\n\n"
    f"Top 5 most suspicious (cmdb_id / kpi_name) samples:\n"
    + "\n".join(top5_lines)
    + "\n\n"
    f"Interpretation guidance: many series are constant or quantized, so simple thresholding against global P95/P05 marks most points as anomalies. "
    f"Focus initial investigation on KPIs with high counts and non-constant variation (e.g., latency-* entries shown lower in the table) for meaningful deviations."
)

# Output the summary string (as variable)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count       p50       p95       p05       max  incident_points  high_anomaly_count  low_anomaly_count    earliest_high_anom_utc     earliest_low_anom_utc
0         checkoutservice      socket     25  9.000000  9.000000  9.000000  9.000000               25                  25                 25 2024-01-17 20:58:00+00:00 2024-01-17 20:58:00+00:00
1         currencyservice      socket     25  4.000000  4.000000  4.000000  4.000000               25                  25                 25 2024-01-17 20:58:00+00:00 2024-01-17 20:58:00+00:00
2   recommendationservice      socket     25  4.000000  4.000000  4.000000  4.000000               25                  25                 25 2024-01-17 20:58:00+00:00 2024-01-17 20:58:00+00:00
3            emailservice      socket     25  3.000000  3.000000  3.000000  3.000000               25                  25                 25 2024-01-17 20:58:00+00:00 2024-01-17 20:58:00+00:00
4   productcatalogservice      socket     25  3.000000  3.000000  3.000000  3.000000               25                  25                 25 2024-01-17 20:58:00+00:00 2024-01-17 20:58:00+00:00
..                    ...         ...    ...       ...       ...       ...       ...              ...                 ...                ...                       ...                       ...
29  recommendationservice  latency-50     25  0.007586  0.043105  0.007367  0.043319               25                   2                  2 2024-01-17 21:13:00+00:00 2024-01-17 20:59:00+00:00
30        currencyservice  latency-90     25  0.029609  0.037924  0.017272  0.043294               25                   2                  2 2024-01-17 21:05:00+00:00 2024-01-17 21:15:00+00:00
31  productcatalogservice  latency-90     25  0.004401  0.004428  0.004379  0.004471               25                   2                  2 2024-01-17 21:11:00+00:00 2024-01-17 21:04:00+00:00
32        currencyservice  latency-50     25  0.003570  0.003666  0.003452  0.003736               25                   2                  2 2024-01-17 21:03:00+00:00 2024-01-17 21:15:00+00:00
33  productcatalogservice  latency-50     25  0.001927  0.002032  0.001829  0.002132               25                   2                  2 2024-01-17 21:11:00+00:00 2024-01-17 21:01:00+00:00

[34 rows x 12 columns],    distinct_pairs_total  pairs_with_high_anomaly_count_gt0  pairs_with_low_anomaly_count_gt0
0                    34                                 34                                34)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse precomputed full results table `final_df` from memory
df = final_df.copy()

# Filter out constant/quantized series where p95 == p05 (use isclose for numeric stability)
non_constant = df[~np.isclose(df['p95'].fillna(0), df['p05'].fillna(0))].copy()

# Keep only KPIs that contain any of the target substrings (case-insensitive)
pattern = r'latency|cpu|mem|diskio|error'
candidates = non_constant[non_constant['kpi_name'].str.contains(pattern, case=False, na=False)].copy()

# Sort by high_anomaly_count desc, then incident_points desc
candidates_sorted = candidates.sort_values(by=['high_anomaly_count', 'incident_points'], ascending=[False, False])

# Select requested columns and limit to top 20 rows
out_cols = [
    'cmdb_id', 'kpi_name', 'incident_points', 'high_anomaly_count', 'low_anomaly_count',
    'earliest_high_anom_utc', 'earliest_low_anom_utc', 'p50', 'p95', 'p05', 'max'
]
filtered_top20 = candidates_sorted[out_cols].head(20).reset_index(drop=True)

# Aggregate summary: number of filtered candidate series and earliest earliest_high_anom_utc among them
num_filtered = len(candidates_sorted)
earliest_high_min = candidates_sorted['earliest_high_anom_utc'].min() if num_filtered > 0 else pd.NaT
agg_summary = pd.DataFrame([{
    'num_filtered_candidate_series': num_filtered,
    'earliest_earliest_high_anom_utc': earliest_high_min
}])

# Return compact table and aggregate summary
filtered_top20, agg_summary

```
Out[3]:
```
```python
# Build a plain-English summary string from in-memory results (filtered_top20, agg_summary)
top = filtered_top20.copy()
agg = agg_summary.iloc[0]

num_filtered = int(agg['num_filtered_candidate_series'])
earliest_high = agg['earliest_earliest_high_anom_utc']
earliest_high_s = earliest_high.isoformat() if pd.notnull(earliest_high) else "N/A"

# Prepare top 5 lines
top5 = top.head(5)
lines = []
for _, r in top5.iterrows():
    ch = r['cmdb_id']
    kpi = r['kpi_name']
    ip = int(r['incident_points'])
    hi = int(r['high_anomaly_count'])
    lo = int(r['low_anomaly_count'])
    eh = r['earliest_high_anom_utc']
    el = r['earliest_low_anom_utc']
    eh_s = eh.isoformat() if pd.notnull(eh) else "N/A"
    el_s = el.isoformat() if pd.notnull(el) else "N/A"
    lines.append(f"- {ch} / {kpi}: incident_points={ip}, high={hi}, low={lo}, first_high={eh_s}, first_low={el_s}")

summary = (
    f"Summary of filtered candidate metric series (non-constant & matching latency/cpu/mem/diskio/error):\n\n"
    f"- Number of filtered candidate series: {num_filtered}\n"
    f"- Earliest high anomaly among them: {earliest_high_s} (UTC)\n\n"
    f"Top 5 series by high_anomaly_count then incident_points:\n"
    + "\n".join(lines)
    + "\n\n"
    f"Note: Most series show only a small number of high anomalies during the incident window (commonly 2), "
    f"and the earliest detected high anomaly across candidates is at {earliest_high_s} UTC. "
    f"Recommendation: investigate the listed top series (e.g., checkoutservice CPU and latency KPIs) first, "
    f"and focus on series with meaningful variation rather than quantized/constant metrics."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  incident_points  high_anomaly_count  low_anomaly_count    earliest_high_anom_utc     earliest_low_anom_utc           p50           p95           p05           max
0         checkoutservice         cpu               25                   2                  2 2024-01-17 21:09:00+00:00 2024-01-17 21:19:00+00:00  4.249743e-01  5.533082e-01  3.103076e-01  6.227734e-01
1         checkoutservice  latency-50               25                   2                  2 2024-01-17 21:05:00+00:00 2024-01-17 21:15:00+00:00  8.217256e-02  1.153139e-01  4.942367e-02  1.458544e-01
2         checkoutservice  latency-90               25                   2                  2 2024-01-17 20:59:00+00:00 2024-01-17 21:20:00+00:00  2.157390e-01  2.302463e-01  2.011443e-01  2.356378e-01
3         checkoutservice         mem               25                   2                  2 2024-01-17 21:05:00+00:00 2024-01-17 21:21:00+00:00  1.194865e+07  1.212914e+07  1.165012e+07  1.218355e+07
4         currencyservice         cpu               25                   2                  2 2024-01-17 20:58:00+00:00 2024-01-17 21:13:00+00:00  3.096870e+01  3.248451e+01  2.977954e+01  3.282706e+01
5         currencyservice  latency-50               25                   2                  2 2024-01-17 21:03:00+00:00 2024-01-17 21:15:00+00:00  3.570149e-03  3.666229e-03  3.451580e-03  3.736457e-03
6         currencyservice  latency-90               25                   2                  2 2024-01-17 21:05:00+00:00 2024-01-17 21:15:00+00:00  2.960882e-02  3.792397e-02  1.727188e-02  4.329355e-02
7         currencyservice         mem               25                   2                  2 2024-01-17 21:08:00+00:00 2024-01-17 21:16:00+00:00  4.455103e+07  4.539022e+07  4.416872e+07  4.553810e+07
8            emailservice         cpu               25                   2                  2 2024-01-17 21:12:00+00:00 2024-01-17 21:20:00+00:00  3.152618e-01  3.509827e-01  2.805087e-01  3.647411e-01
9            emailservice  latency-50               25                   2                  9 2024-01-17 20:58:00+00:00 2024-01-17 21:00:00+00:00  3.025701e-03  3.093972e-03  3.000000e-03  3.213277e-03
10           emailservice  latency-90               25                   2                  9 2024-01-17 20:58:00+00:00 2024-01-17 21:00:00+00:00  4.646262e-03  4.769150e-03  4.600000e-03  5.432708e-03
11           emailservice         mem               25                   2                  2 2024-01-17 21:02:00+00:00 2024-01-17 20:58:00+00:00  4.357004e+07  4.357136e+07  4.356740e+07  4.357222e+07
12  productcatalogservice         cpu               25                   2                  2 2024-01-17 21:00:00+00:00 2024-01-17 21:18:00+00:00  3.424060e+00  3.682600e+00  3.013249e+00  3.842456e+00
13  productcatalogservice  latency-50               25                   2                  2 2024-01-17 21:11:00+00:00 2024-01-17 21:01:00+00:00  1.927490e-03  2.032388e-03  1.829382e-03  2.132381e-03
14  productcatalogservice  latency-90               25                   2                  2 2024-01-17 21:11:00+00:00 2024-01-17 21:04:00+00:00  4.400890e-03  4.427791e-03  4.378784e-03  4.470676e-03
15  productcatalogservice         mem               25                   2                  2 2024-01-17 21:11:00+00:00 2024-01-17 20:58:00+00:00  1.021003e+07  1.045702e+07  1.001076e+07  1.056986e+07
16  recommendationservice         cpu               25                   2                  2 2024-01-17 21:14:00+00:00 2024-01-17 20:59:00+00:00  3.593780e+00  1.973851e+01  2.642593e+00  1.988245e+01
17  recommendationservice      diskio               25                   2                 23 2024-01-17 21:10:00+00:00 2024-01-17 20:58:00+00:00  0.000000e+00  3.806531e+04  0.000000e+00  1.986644e+05
18  recommendationservice  latency-50               25                   2                  2 2024-01-17 21:13:00+00:00 2024-01-17 20:59:00+00:00  7.586007e-03  4.310547e-02  7.367469e-03  4.331872e-02
19  recommendationservice  latency-90               25                   2                  2 2024-01-17 21:12:00+00:00 2024-01-17 21:05:00+00:00  1.617689e-02  9.466757e-02  9.525738e-03  1.015765e-01,    num_filtered_candidate_series earliest_earliest_high_anom_utc
0                             21       2024-01-17 20:58:00+00:00)```
```

In [ ]:
# In[4]:

import re
import pandas as pd
import numpy as np

# Reuse existing 'trace' DataFrame if present; otherwise load and parse
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
if trace['timestamp'].dtype == object or not pd.api.types.is_datetime64_any_dtype(trace['timestamp']):
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per trace_name using the full series
global_p95 = trace.groupby('trace_name', dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# 2) Define incident window (UTC)
start = pd.to_datetime("2024-01-17 20:55:32", utc=True)
end = pd.to_datetime("2024-01-17 21:25:32", utc=True)

# 3) Filter trace rows to the incident window
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 4) Merge global_p95 with the window rows so every trace_name is represented
tw = global_p95.merge(trace_window, on='trace_name', how='left')

# 5) Determine exceedances in-window (value >= global_p95). For rows with no value, treat as no exceed.
tw['exceed'] = False
mask_valid = tw['value'].notna() & tw['global_p95'].notna()
tw.loc[mask_valid, 'exceed'] = tw.loc[mask_valid, 'value'] >= tw.loc[mask_valid, 'global_p95']

# 6) Aggregate per trace_name
agg = tw.groupby(['trace_name', 'global_p95'], dropna=False).agg(
    incident_points = ('value', lambda x: x.notna().sum()),
    exceed_count = ('exceed', 'sum'),
    earliest_exceed_utc = ('timestamp', lambda x: x[tw.loc[x.index, 'exceed']].min() if x.notna().any() else pd.NaT),
    max_value_in_window = ('value', 'max')
).reset_index()

# 7) Extract service name from trace_name if pattern 'from_<service>' or 'to_<service>' exists
# Use regex to capture the service token after from_ or to_
service_extract = agg['trace_name'].str.extract(r'(?:from_|to_)([A-Za-z0-9]+)', expand=False)
agg['service'] = service_extract.fillna('root/other')

# 8) Reorder/select columns as requested
trace_summary = agg[['trace_name', 'service', 'global_p95', 'incident_points', 'exceed_count', 'earliest_exceed_utc', 'max_value_in_window']].copy()

# 9) Sort by exceed_count desc then incident_points desc and limit to top 20
trace_summary_sorted = trace_summary.sort_values(by=['exceed_count', 'incident_points'], ascending=[False, False]).head(20).reset_index(drop=True)

# 10) Aggregate: number of trace_name entries with exceed_count>0 (use full agg, not just top 20)
num_exceeding = int((agg['exceed_count'] > 0).sum())
trace_agg = pd.DataFrame([{
    'trace_names_total': len(agg),
    'trace_names_with_exceed_count_gt0': num_exceeding
}])

# Return compact results
trace_summary_sorted, trace_agg

```
Out[4]:
```
# Build a plain-English summary string from in-memory trace_summary_sorted and trace_agg
import pandas as pd

ts = trace_summary_sorted.copy()
agg = trace_agg.iloc[0]

total = int(agg['trace_names_total'])
with_exceed = int(agg['trace_names_with_exceed_count_gt0'])
earliest_ex = ts['earliest_exceed_utc'].min() if not ts['earliest_exceed_utc'].isnull().all() else pd.NaT
earliest_ex_s = earliest_ex.isoformat() if pd.notnull(earliest_ex) else "N/A"

# Prepare top 6 example lines
examples = []
for _, r in ts.head(6).iterrows():
    tn = r['trace_name']
    sv = r['service']
    gp95 = r['global_p95']
    ip = int(r['incident_points'])
    ex = int(r['exceed_count'])
    ee = r['earliest_exceed_utc']
    mv = r['max_value_in_window']
    ee_s = ee.isoformat() if pd.notnull(ee) else "N/A"
    examples.append(f"- {tn} (service={sv}): global_p95={gp95:.6g}, incident_points={ip}, exceed_count={ex}, earliest_exceed={ee_s}, max_in_window={mv:.6g}")

summary = (
    "Trace telemetry summary for incident window [2024-01-17 20:55:32 UTC, 2024-01-17 21:25:32 UTC]:\n\n"
    f"- Total distinct trace_name entries analyzed: {total}\n"
    f"- Entries with at least one in-window exceedance of global P95: {with_exceed}\n"
    f"- Earliest exceedance among the reported top entries: {earliest_ex_s} (UTC)\n\n"
    "Key observations:\n"
    "1) Many error_rate traces show global P95 = 0 and max = 0. Because the exceed test is value >= global_p95, these zero-series are all counted as 'exceeding' (e.g., trace.from_root.error_rate: 175/175). "
    "Those do not represent meaningful anomalies and should be deprioritized.\n\n"
    "2) More meaningful signals are present in duration and row_count traces. Example top entries:\n"
    + "\n".join(examples)
    + "\n\n"
    "Interpretation / next steps:\n"
    "- Ignore or treat specially the error_rate traces with global_p95 == 0 (they are trivially flagged).\n"
    "- Focus investigation on duration_* and row_count traces (they show non-zero maxima and discrete exceedances). "
    "Notable candidates: root duration/row_count and 'from_checkoutservice' / 'from_frontendservice' duration and row_count traces.\n"
    "- Investigate service dependencies around the earliest exceed times (around 2024-01-17 21:00–21:11 UTC) for root and checkout/frontend traces.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                     trace_name                service   global_p95  incident_points  exceed_count       earliest_exceed_utc  max_value_in_window
0                    trace.from_root.error_rate                   root     0.000000              175           175 2024-01-17 20:58:00+00:00             0.000000
1         trace.from_checkoutservice.error_rate        checkoutservice     0.000000              125           125 2024-01-17 20:58:00+00:00             0.000000
2         trace.from_frontendservice.error_rate        frontendservice     0.000000              125           125 2024-01-17 20:58:00+00:00             0.000000
3     trace.to_productcatalogservice.error_rate  productcatalogservice     0.000000              100           100 2024-01-17 20:58:00+00:00             0.000000
4           trace.to_checkoutservice.error_rate        checkoutservice     0.000000               75            75 2024-01-17 20:58:00+00:00             0.000000
5           trace.to_currencyservice.error_rate        currencyservice     0.000000               75            75 2024-01-17 20:58:00+00:00             0.000000
6     trace.to_recommendationservice.error_rate  recommendationservice     0.000000               75            75 2024-01-17 20:58:00+00:00             0.000000
7   trace.from_recommendationservice.error_rate  recommendationservice     0.000000               50            50 2024-01-17 20:58:00+00:00             0.000000
8              trace.to_emailservice.error_rate           emailservice     0.000000               50            50 2024-01-17 20:58:00+00:00             0.000000
9           trace.to_frontendservice.error_rate        frontendservice     0.000000               50            50 2024-01-17 20:58:00+00:00             0.000000
10           trace.to_paymentservice.error_rate         paymentservice     0.000000               50            50 2024-01-17 20:58:00+00:00             0.000000
11                trace.from_root.duration_mean                   root     0.081855              175             9 2024-01-17 21:11:00+00:00             0.104334
12                 trace.from_root.duration_p95                   root     0.179802              175             9 2024-01-17 21:00:00+00:00             0.242706
13                    trace.from_root.row_count                   root   820.300000              175             9 2024-01-17 21:00:00+00:00           867.000000
14     trace.from_checkoutservice.duration_mean        checkoutservice     0.003891              125             7 2024-01-17 20:58:00+00:00             0.009378
15      trace.from_checkoutservice.duration_p95        checkoutservice     0.005762              125             7 2024-01-17 20:58:00+00:00             0.006140
16         trace.from_checkoutservice.row_count        checkoutservice   267.600000              125             7 2024-01-17 21:00:00+00:00           317.000000
17     trace.from_frontendservice.duration_mean        frontendservice     0.092401              125             7 2024-01-17 20:58:00+00:00             0.136383
18      trace.from_frontendservice.duration_p95        frontendservice     0.195705              125             7 2024-01-17 20:58:00+00:00             0.359367
19         trace.from_frontendservice.row_count        frontendservice  7947.000000              125             7 2024-01-17 20:59:00+00:00          8230.000000,    trace_names_total  trace_names_with_exceed_count_gt0
0                 44                                 44)```
```

In [ ]:
# In[5]:

# Reuse aggregated trace summary `trace_summary` from memory (created earlier)
# Filter entries: exceed_count>0 AND global_p95>0 and service in target services
targets = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

filtered = trace_summary[
    (trace_summary['exceed_count'] > 0) &
    (trace_summary['global_p95'] > 0) &
    (trace_summary['service'].isin(targets))
].copy()

# Ensure earliest_exceed_utc is datetime (it should be) and sort ascending by it
filtered['earliest_exceed_utc'] = pd.to_datetime(filtered['earliest_exceed_utc'], utc=True, errors='coerce')
filtered_sorted = filtered.sort_values(by=['earliest_exceed_utc'], ascending=True)

# Select requested columns and limit to top 20
out_cols = ['trace_name', 'service', 'global_p95', 'incident_points', 'exceed_count', 'earliest_exceed_utc', 'max_value_in_window']
filtered_top20 = filtered_sorted[out_cols].head(20).reset_index(drop=True)

# Aggregate single-line count
agg_count = pd.DataFrame([{
    'filtered_trace_entries_count': int(len(filtered))
}])

# Return compact table and aggregate count
filtered_top20, agg_count

```
Out[5]:
```
Summary (plain English)

- Filtered trace entries (exceed_count > 0 and global_p95 > 0) for the target services: 21 trace_name entries.

- Earliest exceedance among these entries: 2024-01-17T20:58:00+00:00 (UTC).

Top affected trace entries (examples, sorted by earliest exceed time)
- trace.from_checkoutservice.duration_mean (service=checkoutservice): global_p95=0.003891, incident_points=125, exceed_count=7, earliest_exceed=2024-01-17T20:58:00Z, max_in_window=0.009378
- trace.from_checkoutservice.duration_p95 (checkoutservice): global_p95=0.005762, incident_points=125, exceed_count=7, earliest_exceed=2024-01-17T20:58:00Z, max_in_window=0.006140
- trace.to_productcatalogservice.duration_mean (productcatalogservice): global_p95=0.004292, incident_points=100, exceed_count=5, earliest_exceed=2024-01-17T20:58:00Z, max_in_window=0.004981
- trace.to_checkoutservice.duration_mean (checkoutservice): global_p95=0.098336, incident_points=75, exceed_count=4, earliest_exceed=2024-01-17T20:58:00Z, max_in_window=0.136383
- trace.to_checkoutservice.duration_p95 (checkoutservice): global_p95=0.207335, incident_points=75, exceed_count=4, earliest_exceed=2024-01-17T20:58:00Z, max_in_window=0.359367
- trace.to_currencyservice.row_count (currencyservice): global_p95=2379.2, incident_points=75, exceed_count=4, earliest_exceed=2024-01-17T20:59:00Z, max_in_window=2426
- trace.from_checkoutservice.row_count (checkoutservice): global_p95=267.6, incident_points=125, exceed_count=7, earliest_exceed=2024-01-17T21:00:00Z, max_in_window=317
- trace.to_productcatalogservice.row_count (productcatalogservice): global_p95=3750.25, incident_points=100, exceed_count=5, earliest_exceed=2024-01-17T21:01:00Z, max_in_window=3816
- trace.to_emailservice.duration_p95 (emailservice): global_p95=0.000631, incident_points=50, exceed_count=3, earliest_exceed=2024-01-17T21:04:00Z, max_in_window=0.000903
- trace.to_currencyservice.duration_p95 (currencyservice): global_p95=0.000249, incident_points=75, exceed_count=4, earliest_exceed=2024-01-17T21:04:00Z, max_in_window=0.030278
(These are representative; the full filtered list contains 21 entries.)

Interpretation and suggested next steps
- These non-trivial exceedances concentrate on duration and row_count traces for checkoutservice, productcatalogservice, currencyservice, recommendationservice, and emailservice. The earliest deviations appear at ~2024-01-17T20:58:00Z.
- Prioritize investigating checkoutservice (multiple duration and row_count traces with 7 exceedances and early timestamps) and productcatalogservice/currencyservice traces next — they show meaningful exceedances of their global P95 thresholds.
- Use the listed trace_name rows and earliest_exceed timestamps to correlate with service logs, metrics (CPU/memory/latency), and upstream/downstream calls to find root cause (load spikes, increased latency, or service degradation).

The original code execution output of IPython Kernel is also provided below for reference:

(                                        trace_name                service   global_p95  incident_points  exceed_count       earliest_exceed_utc  max_value_in_window
0         trace.from_checkoutservice.duration_mean        checkoutservice     0.003891              125             7 2024-01-17 20:58:00+00:00             0.009378
1          trace.from_checkoutservice.duration_p95        checkoutservice     0.005762              125             7 2024-01-17 20:58:00+00:00             0.006140
2     trace.to_productcatalogservice.duration_mean  productcatalogservice     0.004292              100             5 2024-01-17 20:58:00+00:00             0.004981
3           trace.to_checkoutservice.duration_mean        checkoutservice     0.098336               75             4 2024-01-17 20:58:00+00:00             0.136383
4            trace.to_checkoutservice.duration_p95        checkoutservice     0.207335               75             4 2024-01-17 20:58:00+00:00             0.359367
5               trace.to_currencyservice.row_count        currencyservice  2379.200000               75             4 2024-01-17 20:59:00+00:00          2426.000000
6             trace.from_checkoutservice.row_count        checkoutservice   267.600000              125             7 2024-01-17 21:00:00+00:00           317.000000
7         trace.to_productcatalogservice.row_count  productcatalogservice  3750.250000              100             5 2024-01-17 21:01:00+00:00          3816.000000
8               trace.to_emailservice.duration_p95           emailservice     0.000631               50             3 2024-01-17 21:04:00+00:00             0.000903
9              trace.to_emailservice.duration_mean           emailservice     0.000334               50             3 2024-01-17 21:04:00+00:00             0.000358
10           trace.to_currencyservice.duration_p95        currencyservice     0.000249               75             4 2024-01-17 21:04:00+00:00             0.030278
11          trace.to_currencyservice.duration_mean        currencyservice     0.000912               75             4 2024-01-17 21:06:00+00:00             0.005648
12      trace.from_recommendationservice.row_count  recommendationservice   582.000000               50             4 2024-01-17 21:07:00+00:00           582.000000
13        trace.to_recommendationservice.row_count  recommendationservice   581.300000               75             4 2024-01-17 21:07:00+00:00           582.000000
14              trace.to_checkoutservice.row_count        checkoutservice   290.900000               75             4 2024-01-17 21:08:00+00:00           317.000000
15                 trace.to_emailservice.row_count           emailservice    36.000000               50             4 2024-01-17 21:08:00+00:00            37.000000
16     trace.to_productcatalogservice.duration_p95  productcatalogservice     0.011396              100             5 2024-01-17 21:11:00+00:00             0.014095
17   trace.from_recommendationservice.duration_p95  recommendationservice     0.082388               50             3 2024-01-17 21:12:00+00:00             0.095090
18  trace.from_recommendationservice.duration_mean  recommendationservice     0.017030               50             3 2024-01-17 21:12:00+00:00             0.022019
19    trace.to_recommendationservice.duration_mean  recommendationservice     0.020998               75             4 2024-01-17 21:12:00+00:00             0.026979,    filtered_trace_entries_count
0                            21)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing `log` DataFrame if present; otherwise load and parse timestamps
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed correctly
if not pd.api.types.is_datetime64_any_dtype(log['timestamp']):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# Restrict to candidate services
log_candidates = log[log['cmdb_id'].isin(candidates)].copy()

# Compute global P95 per (cmdb_id, log_name) using the full series (before time filtering)
global_p95 = (
    log_candidates
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 20:55:32", utc=True)
end = pd.to_datetime("2024-01-17 21:25:32", utc=True)

# Filter log rows to the incident window
log_window = log_candidates[(log_candidates['timestamp'] >= start) & (log_candidates['timestamp'] <= end)].copy()

# Merge global thresholds with window rows (left join from global_p95 to keep all pairs)
merged = global_p95.merge(log_window, on=['cmdb_id', 'log_name'], how='left')

# Determine exceedances: value >= global_p95 (only where value is present)
merged['exceed'] = False
valid_mask = merged['value'].notna() & merged['global_p95'].notna()
merged.loc[valid_mask, 'exceed'] = merged.loc[valid_mask, 'value'] >= merged.loc[valid_mask, 'global_p95']

# Aggregate per (cmdb_id, log_name)
agg = (
    merged
    .groupby(['cmdb_id', 'log_name', 'global_p95'], dropna=False)
    .agg(
        incident_points = ('value', lambda x: int(x.notna().sum())),
        exceed_count = ('exceed', 'sum'),
        max_in_window = ('value', 'max')
    )
    .reset_index()
)

# Ensure exceed_count is int
agg['exceed_count'] = agg['exceed_count'].fillna(0).astype(int)

# Earliest exceed timestamp per pair (if any)
earliest_exceed = (
    merged[merged['exceed']]
    .groupby(['cmdb_id', 'log_name'])['timestamp']
    .min()
    .reset_index(name='earliest_exceed_utc')
)

# Merge earliest exceed into agg
agg = agg.merge(earliest_exceed, on=['cmdb_id', 'log_name'], how='left')

# Sort by exceed_count desc then incident_points desc and limit to top 20
agg_sorted = agg.sort_values(by=['exceed_count', 'incident_points'], ascending=[False, False]).reset_index(drop=True)
top20 = agg_sorted.head(20)[['cmdb_id', 'log_name', 'global_p95', 'incident_points', 'exceed_count', 'earliest_exceed_utc', 'max_in_window']].copy()

# Aggregate single-line: number of pairs with exceed_count>0
num_pairs_exceed = int((agg['exceed_count'] > 0).sum())
agg_count = pd.DataFrame([{'pairs_with_exceed_count_gt0': num_pairs_exceed}])

# Return compact results
top20, agg_count

```
Out[6]:
```
```python
# Build a plain-English summary string from in-memory results (top20, agg_count)
import pandas as pd

tb = top20.copy()
agg = int(agg_count.at[0, 'pairs_with_exceed_count_gt0'])

# Prepare up to 6 example lines
lines = []
for _, r in tb.head(6).iterrows():
    cm = r['cmdb_id']
    ln = r['log_name']
    gp95 = r['global_p95']
    ip = int(r['incident_points'])
    ex = int(r['exceed_count'])
    ee = r['earliest_exceed_utc']
    mv = r['max_in_window']
    ee_s = ee.isoformat() if pd.notnull(ee) else "N/A"
    lines.append(f"- {cm} / {ln}: global_p95={gp95}, incident_points={ip}, exceed_count={ex}, earliest_exceed={ee_s}, max_in_window={mv}")

summary = (
    "Log telemetry summary for the incident window [2024-01-17 20:55:32 UTC to 2024-01-17 21:25:32 UTC]:\n\n"
    f"- Number of (cmdb_id, log_name) pairs with exceed_count>0: {agg}\n\n"
    "Top examples:\n"
    + "\n".join(lines)
    + "\n\n"
    "Notes:\n"
    "- Several log.error_count series have global P95 = 0 and are therefore trivially flagged (25/25 points); these are not necessarily meaningful errors.\n"
    "- Non-trivial signals appear in log.row_count for multiple services (checkoutservice, currencyservice, emailservice, recommendationservice) where exceedances correspond to increased row counts; investigate those timestamps (earliest examples listed above) for workload spikes or downstream/backpressure issues."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  global_p95  incident_points  exceed_count       earliest_exceed_utc  max_in_window
0        checkoutservice  log.error_count         0.0               25            25 2024-01-17 20:58:00+00:00              0
1        currencyservice  log.error_count         0.0               25            25 2024-01-17 20:58:00+00:00              0
2           emailservice  log.error_count         0.0               25            25 2024-01-17 20:58:00+00:00              0
3  recommendationservice  log.error_count         0.0               25            25 2024-01-17 20:58:00+00:00              0
4        checkoutservice    log.row_count       108.0               25             4 2024-01-17 21:08:00+00:00            111
5           emailservice    log.row_count        36.0               25             4 2024-01-17 21:08:00+00:00             37
6  recommendationservice    log.row_count       581.0               25             4 2024-01-17 21:00:00+00:00            582
7        currencyservice    log.row_count      2516.0               25             2 2024-01-17 20:59:00+00:00           2521,    pairs_with_exceed_count_gt0
0                            8)```
```